# Expense analyzer — findings

**Fill this cell in last, and present only this cell.**

1. _`<Category>` is Rs `<x>`/month, `<y>`% of spend._
2. _`<z>`% of `<category>` spending happens at weekends._
3. _Spend rose `<n>`% from `<month>` to `<month>`, driven by `<category>`._
4. _Median transaction is Rs `<m>` — bled by frequency, not size._
5. _Savings rate is `<a>`%; cutting `<category>` would take it to `<b>`%._

**What did not work:** _`<the rule that misfired, and why>`_

In [ ]:
%load_ext autoreload
%autoreload 2

import sys

sys.path.append("..") if "src" not in sys.path else None

import pandas as pd

from src import analyze, categorize
from src.clean import load_and_clean

pd.set_option("display.max_colwidth", 60)

## 1. Load and clean

Look at the raw frame before trusting anything: `dtypes` shows the amount
columns arrive as `object` (text), not numbers.

In [ ]:
frame = load_and_clean("data/sample_statement.csv")
print(frame.shape)
frame.head()

## 2. Categorize, then check coverage

Read the unmatched narrations, add keywords to `CATEGORY_RULES`, re-run.
Repeat until uncategorised is under 5% **by value**.

In [ ]:
frame = categorize.add_categories(frame)
categorize.coverage(frame)

In [ ]:
categorize.unmatched_narrations(frame)

## 3. Aggregate

In [ ]:
spend = analyze.spending_only(frame)

print(analyze.by_category(spend).round(0))
print()
print(analyze.monthly_totals(spend).round(0))
print()
print(f"savings rate: {analyze.savings_rate(frame):.1%}")
print(f"median transaction: {spend['amount'].median():,.0f}")

In [ ]:
analyze.recurring_candidates(spend)

## 4. Charts

Inline here; `run.py` writes the same four to `reports/`.

In [ ]:
%matplotlib inline

analyze.by_category(spend).sort_values().plot.barh(
    figsize=(9, 5), xlabel="Rupees spent", title="Spend by category"
);

In [ ]:
analyze.category_by_month(spend).plot.bar(
    stacked=True, figsize=(9, 5), title="Spend by category, per month"
);

## 5. Weekend vs weekday

Statements have no clock time, so day-of-week is the finest time signal
available. Hour-of-day analysis is not possible from this data.

In [ ]:
food = spend[spend["category"] == "Food"]
share = food[food["is_weekend"]]["amount"].sum() / food["amount"].sum()
print(f"{share:.0%} of food spending falls at the weekend")

analyze.weekday_split(spend).plot.bar(figsize=(9, 4), title="Spend by day of week");